# S9 - Sistemas expertos (Pagarium) — Soluciones

Cuaderno **de soluciones** de la práctica guiada de la S9 (RA5-a/c). Está resuelto: cada celda muestra la **salida real**. Se corresponde con el cuaderno de enunciados `sesion09_sistemas_expertos.ipynb`.

> Hilo conductor: **Pagarium**, una pasarela de pagos que decide si aprueba o rechaza una transacción.

## 1. El micro-motor que falla (reproducir el bug)

El motor dispara **todas** las reglas que encajan, sin resolver conflictos. Resultado: para el mismo pago deduce **primero "aprobar" y luego "rechazar"** — dos decisiones contradictorias.

In [1]:
class MicroMotor:
    def __init__(self):
        self.hechos, self.reglas, self.traza = {}, [], []

    def hecho(self, **h):
        self.hechos.update(h)

    def regla(self, nombre, prioridad, condicion, accion):
        self.reglas.append((nombre, prioridad, condicion, accion))

    def run(self):
        pendientes = True
        while pendientes:
            pendientes = False
            for nombre, prioridad, condicion, accion in self.reglas:
                if nombre in self.traza:
                    continue
                if condicion(self.hechos):          # match
                    self.traza.append(nombre)        # (sin resolve)
                    accion(self.hechos)              # act
                    pendientes = True
        return self.hechos

motor = MicroMotor()
motor.hecho(importe=200, antiguedad_meses=10, n_intentos=5)
motor.regla("aprobar_pequena", 10, lambda h: h["importe"] < 500,
            lambda h: h.update(decision="aprobar"))
motor.regla("rechazar_reintentos", 10, lambda h: h["n_intentos"] >= 4,
            lambda h: h.update(decision="rechazar"))
print(motor.run())
print(motor.traza)

{'importe': 200, 'antiguedad_meses': 10, 'n_intentos': 5, 'decision': 'rechazar'}
['aprobar_pequena', 'rechazar_reintentos']


El bug tiene **dos causas**:

1. **No hay `resolve`**: dispara todo lo que encaja, en el orden en que se declaró.
2. **Las reglas no son excluyentes**: `aprobar` solo mira `importe < 500`; no exige `n_intentos < 4`. Por eso, con `n_intentos = 5`, ambas reglas encajan a la vez.

La solución toca las dos cosas.

### Otro dominio: control de calefacción

El mismo micro-motor sirve para cualquier dominio; solo cambian las reglas y los hechos.

In [2]:
# El mismo micro-motor, otro dominio: calefacción
motor = MicroMotor()
motor.hecho(temperatura=18, hora=23, ventana_abierta=False)
motor.regla("frio", 10, lambda h: h["temperatura"] < 20, lambda h: h.update(encender=True))
motor.regla("noche", 10, lambda h: h["hora"] >= 22, lambda h: h.update(modo="eco"))
motor.regla("ventilar", 5, lambda h: h["ventana_abierta"], lambda h: h.update(encender=False))
print(motor.run())
print(motor.traza)

{'temperatura': 18, 'hora': 23, 'ventana_abierta': False, 'encender': True, 'modo': 'eco'}
['frio', 'noche']


## 2. Solución: resolución de conflictos con `salience`

En `run()` añadimos la fase **resolver**: si hay varias reglas activas, se dispara **solo la de mayor prioridad**. Y corregimos `aprobar` para que sea **mutuamente excluyente** con `rechazar` (`importe < 500` **y** `n_intentos < 4`), como en la versión de `experta`.

In [3]:
class MicroMotor2:
    def __init__(self):
        self.hechos, self.reglas, self.traza = {}, [], []

    def hecho(self, **h):
        self.hechos.update(h)

    def regla(self, nombre, prioridad, condicion, accion):
        self.reglas.append((nombre, prioridad, condicion, accion))

    def run(self):
        while True:
            # 1) reconocer (match): reglas que encajan y aún no se han disparado
            activas = [(p, n, a) for n, p, c, a in self.reglas
                       if n not in self.traza and c(self.hechos)]
            if not activas:
                break
            # 2) resolver (resolve): solo la de MAYOR prioridad (salience)
            _, nombre, accion = max(activas, key=lambda t: t[0])
            # 3) actuar (act)
            self.traza.append(nombre)
            accion(self.hechos)
        return self.hechos

motor = MicroMotor2()
motor.hecho(importe=200, antiguedad_meses=10, n_intentos=5)

# "aprobar" ahora es excluyente con "rechazar": importe < 500 Y n_intentos < 4
motor.regla("aprobar_pequena", 20,
            lambda h: h["importe"] < 500 and h["n_intentos"] < 4,
            lambda h: h.update(decision="aprobar"))
motor.regla("rechazar_reintentos", 30,   # mayor salience: gana si hay conflicto
            lambda h: h["n_intentos"] >= 4,
            lambda h: h.update(decision="rechazar"))

print(motor.run())
print(motor.traza)

{'importe': 200, 'antiguedad_meses': 10, 'n_intentos': 5, 'decision': 'rechazar'}
['rechazar_reintentos']


Con `importe = 200` y `n_intentos = 5` solo encaja `rechazar_reintentos` (porque `aprobar` exige `n_intentos < 4`): la decisión es **`rechazar`**, sin contradicción, y la traza tiene **una sola regla**.

> Si probáramos `n_intentos = 2`, encajaría `aprobar_pequena` y la decisión sería `aprobar`. La `salience` (30 > 20) solo actúa como red de seguridad cuando dos reglas encajan a la vez.

## 3. La misma política con `experta`

`experta` (docencia) ya trae el ciclo reconocer-resolver-actuar y la `salience`. **En Python 3.10+ necesita el parche** de `collections.Mapping` antes de importar. En Colab, instala primero con `%pip install experta`.

In [4]:
import collections, collections.abc
if not hasattr(collections, "Mapping"):
    collections.Mapping = collections.abc.Mapping
    collections.Iterable = collections.abc.Iterable
    collections.MutableMapping = collections.abc.MutableMapping

from experta import *

class Pagarium(KnowledgeEngine):
    @DefFacts()
    def inicio(self):
        yield Fact(accion="evaluar")

    @Rule(Fact(accion="evaluar"), Fact(importe=P(lambda i: i < 500)),
          Fact(n_intentos=P(lambda n: n < 4)), salience=20)
    def aprobar(self):
        self.declare(Fact(decision="aprobar"))

    @Rule(Fact(n_intentos=P(lambda n: n >= 4)), salience=30)
    def rechazar(self):
        self.declare(Fact(decision="rechazar"))

motor = Pagarium(); motor.reset()
motor.declare(Fact(importe=200, n_intentos=5))
motor.run()
print([dict(f) for f in motor.facts.values() if "decision" in f])

[{'decision': 'rechazar', '__factid__': 3}]


### Otro dominio: clasificación de animales

`experta` con reglas de un zoólogo (encadenamiento hacia delante): de los hechos al diagnóstico.

In [5]:
class Animales(KnowledgeEngine):
    @DefFacts()
    def inicio(self):
        yield Fact(analizar=True)

    @Rule(Fact(analizar=True), salience=10)
    def datos(self):
        self.declare(Fact(pelo=True), Fact(carnivoro=True), Fact(color="leonado"), Fact(manchas="oscuras"))

    @Rule(Fact(pelo=True))
    def mamifero(self):
        self.declare(Fact(mamifero=True))

    @Rule(Fact(mamifero=True), Fact(carnivoro=True), Fact(color="leonado"), Fact(manchas="oscuras"))
    def guepardo(self):
        print("IDENTIFICADO: guepardo")

engine = Animales(); engine.reset(); engine.run()

IDENTIFICADO: guepardo


## 4. La misma política en CLIPS (`clipspy`)

`clipspy` es el *binding* de **CLIPS 6.4** (producción). Las reglas se escriben en sintaxis LISP con `(declare (salience 30))`. En Colab: `%pip install clipspy`.

In [6]:
import clips

env = clips.Environment()
env.build("(defrule aprobar (importe ?i&:(< ?i 500)) (n_intentos ?n&:(< ?n 4)) => (assert (decision aprobar)))")
env.build("(defrule rechazar (declare (salience 30)) (n_intentos ?n&:(>= ?n 4)) => (assert (decision rechazar)))")
env.assert_string("(importe 200)")
env.assert_string("(n_intentos 5)")
env.run()
print([str(f) for f in env.facts()])

['(importe 200)', '(n_intentos 5)', '(decision rechazar)']


### Más reglas en CLIPS

Dos reglas que deciden la tarifa según el importe.

In [7]:
env = clips.Environment()
env.build("(defrule pequena (importe ?i&:(< ?i 100)) => (assert (tarifa basica)))")
env.build("(defrule grande (importe ?i&:(>= ?i 100)) => (assert (tarifa premium)))")
env.assert_string("(importe 150)")
env.run()
print([str(f) for f in env.facts()])

['(importe 150)', '(tarifa premium)']


## Cierre: comparativa

| Motor | Resolución de conflictos | Salida con `n_intentos = 5` |
|---|---|---|
| `MicroMotor` (con bug) | Ninguna (dispara todo) | `aprobar` **y** `rechazar` (contradicción) |
| `MicroMotor2` (corregido) | `salience` (mayor prioridad) | solo `rechazar` |
| `experta` | `salience` + reglas excluyentes | solo `rechazar` |
| `clipspy` (CLIPS) | `salience` + reglas excluyentes | solo `rechazar` |

**En 3 líneas:** el motor de producción elige entre las reglas que encajan (resolver) con la `salience`; además, las reglas deben ser **mutuamente excluyentes** (`importe < 500` **y** `n_intentos < 4`) para que no haya dos decisiones. Ese es el papel de la **resolución de conflictos**: garantizar una única conclusión por caso.